In [1]:
import torch
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import gpytorch
import xarray as xr
import matplotlib.pyplot as plt

# Load ice velocity nc file

**antarctic_ice_vel_phase_map_v01.nc** is 6.9 GB so we only use slices which are 100 MB each.

In [2]:
"""
### Load original dataset ###
# Load from whereever it is stored
vel = xr.open_dataset("antarctic_ice_vel_phase_map_v01.nc")

### DOMAINS ###
scene_size = 22500 # in meters
n_scenes = 30 # scenes per row/column: 900 scences per domain
span = scene_size * n_scenes

# Transantarctic mountains: Nimrod, Byrd, Skelton glacier, Victoria land
# mountainous domain spanning grounded ice and floating ice.
# stick to order: y, x
transant_y_min = - 1232000
transant_x_min = - 0
transant_y_max = transant_y_min + span # -557000
transant_x_max = transant_x_min + span # 675000

# Dome C
# lake Vostok is still further north than this domain
domec_y_min = -1232000
domec_x_min = 899000
domec_y_max = domec_y_min + span # -557000
domec_x_max = domec_x_min + span # 1574000

### Create slices ###
# reduces data size down to managable level
# y slicing ordering is (max, min)
vel_transant_slice = vel.sel(x = slice(transant_x_min, transant_x_max), y = slice(transant_y_max, transant_y_min))
vel_domec_slice = vel.sel(x = slice(domec_x_min, domec_x_max), y = slice(domec_y_max, domec_y_min))

### Save ###

vel_transant_slice.to_netcdf(path = './nc_data/antarctic_ice_vel_phase_map_v01_TransantarcticMountains_slice.nc')
vel_domec_slice.to_netcdf(path = './nc_data/antarctic_ice_vel_phase_map_v01_DomeC_slice.nc')
"""

'\n### Load original dataset ###\n# Load from whereever it is stored\nvel = xr.open_dataset("antarctic_ice_vel_phase_map_v01.nc")\n\n### DOMAINS ###\nscene_size = 22500 # in meters\nn_scenes = 30 # scenes per row/column: 900 scences per domain\nspan = scene_size * n_scenes\n\n# Transantarctic mountains: Nimrod, Byrd, Skelton glacier, Victoria land\n# mountainous domain spanning grounded ice and floating ice.\n# stick to order: y, x\ntransant_y_min = - 1232000\ntransant_x_min = - 0\ntransant_y_max = transant_y_min + span # -557000\ntransant_x_max = transant_x_min + span # 675000\n\n# Dome C\n# lake Vostok is still further north than this domain\ndomec_y_min = -1232000\ndomec_x_min = 899000\ndomec_y_max = domec_y_min + span # -557000\ndomec_x_max = domec_x_min + span # 1574000\n\n### Create slices ###\n# reduces data size down to managable level\n# y slicing ordering is (max, min)\nvel_transant_slice = vel.sel(x = slice(transant_x_min, transant_x_max), y = slice(transant_y_max, transant_

In [3]:
vel_transant_slice = xr.open_dataset('./nc_data/antarctic_ice_vel_phase_map_v01_TransantarcticMountains_slice.nc')
vel_domec_slice = xr.open_dataset('./nc_data/antarctic_ice_vel_phase_map_v01_DomeC_slice.nc')

In [4]:
vel_transant_scene = vel_transant_slice.isel(y = slice(0, 50), x = slice(0, 50))

dims = 50

scene_vel_tensor = torch.cat((torch.tensor(vel_transant_scene.VX.values).unsqueeze(0), 
                              torch.tensor(vel_transant_scene.VY.values).unsqueeze(0),
                              torch.tensor(vel_transant_scene.coords["y"].values).unsqueeze(-1).repeat(1, dims).unsqueeze(0),
                              torch.tensor(vel_transant_scene.coords["x"].values).repeat(dims, 1).unsqueeze(0)),
                              dim = 0)

# Save a single scene 50 x 50
torch.save(scene_vel_tensor, './torch_data/scene_vel_tensor.pt')

In [5]:
vel_transant_scene

<xarray.Dataset>
Dimensions:       (x: 50, y: 50)
Coordinates:
  * x             (x) float64 350.0 800.0 1.25e+03 ... 2.195e+04 2.24e+04
  * y             (y) float64 -5.57e+05 -5.574e+05 ... -5.786e+05 -5.79e+05
    lat           (y, x) float64 ...
    lon           (y, x) float64 ...
Data variables:
    coord_system  |S1 ...
    VX            (y, x) float32 2.623 2.044 2.039 ... -30.77 -33.77 -36.13
    VY            (y, x) float32 -1.535 0.5726 1.867 ... -15.71 -13.4 -9.852
    STDX          (y, x) float32 ...
    STDY          (y, x) float32 ...
    ERRX          (y, x) float32 ...
    ERRY          (y, x) float32 ...
    CNT           (y, x) int32 ...
    SOURCE        (y, x) int8 ...
Attributes: (12/27)
    Conventions:               CF-1.6
    Metadata_Conventions:      CF-1.6, Unidata Dataset Discovery v1.0, GDS v2.0
    standard_name_vocabulary:  CF Standard Name Table (v22, 12 February 2013)
    id:                        v_mix.v8Jul2019.nc
    title:                     MEaSURES Antarctica Ice Velocity Map 450m spacing
    product_version:            
    ...                        ...
    time_coverage_start:       1995-01-01
    time_coverage_end:         2016-12-31
    project:                   NASA/MEaSUREs
    creator_name:              J. Mouginot
    comment:                    
    license:                   No restrictions on access or use.